# Install Dependencies

In [1]:
!pip install evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.4 MB/s eta 0:00:0000:0100:01


# Get Dataset

In [2]:
!git clone https://huggingface.co/datasets/Sparkplugx1904/Balinese-Common-Voice/ temp
!mv temp/* ./
!rm -rf temp

Cloning into 'temp'...
remote: Enumerating objects: 2081, done.
remote: Total 2081 (delta 0), reused 0 (delta 0), pack-reused 2081 (from 1)
Receiving objects: 100% (2081/2081), 512.63 KiB | 5.34 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Filtering content: 100% (2073/2073), 2.37 GiB | 154.51 MiB/s, done.


# Import Library

In [3]:
from datasets import load_dataset, Audio, Features, Value, Dataset
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor
import torch
import pandas as pd
import librosa

# Data Loading and Processing

In [4]:
import pandas as pd
from datasets import Dataset, Value

# 1. Load metadata
metadata_df = pd.read_csv("metadata.tsv", sep='\t')

# 2. Gunakan kolom 'path' dan 'balinese' sebagai target transkripsi
metadata_df = metadata_df[["path", "balinese"]].rename(columns={"balinese": "sentence"})

# 3. Pastikan path sudah ada prefix "clips/"
if not metadata_df["path"].str.startswith("clips/").all():
    metadata_df["path"] = "clips/" + metadata_df["path"].astype(str)

# 4. Drop baris dengan nilai kosong
metadata_df = metadata_df.dropna(subset=["path", "sentence"]).reset_index(drop=True)

# 5. Split train / test
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(metadata_df, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Total data: {len(metadata_df)}")
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

# 6. Konversi ke Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset  = Dataset.from_pandas(test_df,  preserve_index=False)

# 7. Cast kolom path sebagai string
train_dataset = train_dataset.cast_column("path", Value("string"))
test_dataset  = test_dataset.cast_column("path",  Value("string"))

print(train_dataset)
print(test_dataset)

Total data: 2071
Train: 1863 | Test: 208


Casting the dataset:   0%|          | 0/1863 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/208 [00:00<?, ? examples/s]

Dataset({
    features: ['path', 'sentence'],
    num_rows: 1863
})
Dataset({
    features: ['path', 'sentence'],
    num_rows: 208
})


# Load Whisper Processor and Dataset

In [5]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

# 1. Muat Processor
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-medium")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-medium", language="indonesian", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-medium", language="indonesian", task="transcribe")

# 2. Tokenisasi label saja — path TETAP ADA untuk audio on-the-fly di collator
def prepare_dataset(batch):
    batch["labels"] = processor.tokenizer(batch["sentence"], truncation=True).input_ids
    return batch

print("Melakukan tokenisasi dataset...")
train_dataset = train_dataset.map(prepare_dataset, remove_columns=["sentence"])
test_dataset  = test_dataset.map(prepare_dataset,  remove_columns=["sentence"])

print(f"Kolom yang tersedia sekarang: {train_dataset.column_names}")
# Output harusnya: ['path', 'labels']

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

Melakukan tokenisasi dataset...


Map:   0%|          | 0/1863 [00:00<?, ? examples/s]

Map:   0%|          | 0/208 [00:00<?, ? examples/s]

Kolom yang tersedia sekarang: ['path', 'labels']


# Load a Pre-Trained Checkpoint

In [6]:
from transformers import WhisperForConditionalGeneration
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")

# Nonaktifkan forced_decoder_ids agar model belajar dari data
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens    = []

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

# Collator

In [7]:
import librosa
import numpy as np
import torch

class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor):
        self.processor = processor
        self.sr = 16000
        self.pad = processor.tokenizer.pad_token_id

    def __call__(self, features):
        # 1. Feature Extraction (On-the-fly) — load audio langsung dari path
        input_features = [{
            "input_features": self.processor.feature_extractor(
                librosa.load(feature["path"], sr=self.sr)[0],
                sampling_rate=self.sr
            ).input_features[0]
        } for feature in features]

        # 2. Padding audio features
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 3. Padding labels
        labels = [torch.tensor(feature["labels"]) for feature in features]
        labels_padded = torch.nn.utils.rnn.pad_sequence(
            labels, batch_first=True, padding_value=self.pad
        )

        batch["labels"] = labels_padded
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# Metric (WER)

In [8]:
import evaluate

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    # Ganti -100 kembali ke pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("Metric WER siap.")

Metric WER siap.


# Launch

In [ ]:
import os
import gc
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, TrainerCallback

# --- 1. SUPER AGGRESSIVE CLEANUP ---
torch.cuda.empty_cache()
gc.collect()

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- 2. CONFIGURATION ---
model.config.use_cache = False

# --- 3. TRAINING ARGUMENTS ---
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-balinese",

    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=2,

    # --- Optimization ---
    learning_rate=1e-5,
    warmup_steps=100,
    num_train_epochs=30,
    lr_scheduler_type="cosine",
    weight_decay=0.01,

    # --- Memory Safety ---
    fp16=True,
    gradient_checkpointing=True,       # Efisiensi memori tanpa turunkan akurasi

    # --- Evaluation dengan WER ---
    predict_with_generate=True,        # Wajib True agar WER bisa dihitung
    generation_max_length=225,
    eval_accumulation_steps=1,

    # --- Logging & Strategy ---
    eval_strategy="steps",
    logging_steps=5,
    save_strategy="steps",
    save_steps=1000,
    eval_steps=1000,
    report_to=["tensorboard"],

    load_best_model_at_end=True,
    metric_for_best_model="wer",       # WER lebih relevan untuk ASR
    greater_is_better=False,

    save_total_limit=1,
    remove_unused_columns=False,
    dataloader_num_workers=2,
)

# --- 4. CALLBACK PEMBERSIH MEMORI ---
class ClearMemoryCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 20 == 0:
            torch.cuda.empty_cache()
            gc.collect()

# --- 5. INITIALIZE TRAINER ---
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[ClearMemoryCallback()]
)

print("Starting Training...")
print(f"Learning Rate: {training_args.learning_rate}")

# --- 6. TRAIN ---
trainer.train()

2026-03-10 04:14:22.192909: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773116062.403446      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773116062.460113      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773116062.932418      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773116062.932466      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773116062.932474      55 computation_placer.cc:177] computation placer alr

Starting Training...
Learning Rate: 1e-05


Step,Training Loss,Validation Loss


In [ ]:
trainer.save_model("./whisper-balinese-final")

In [ ]:
processor.save_pretrained("./whisper-balinese-final")
tokenizer.save_pretrained("./whisper-balinese-final")